# Dense Face Mesh with UniFace

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yakhyo/uniface/blob/main/examples/15_face_mesh.ipynb) [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/code/yakhyokhuja/dense-face-mesh-with-uniface)

MediaPipe Face Mesh fits 468 dense 3D landmarks to a face, or 478 with the irises.
Unlike the other landmark models it returns **three** coordinates per point and a face-presence score, and it runs every face in an image through a **single batched inference call**.

<img src="https://raw.githubusercontent.com/yakhyo/uniface/main/assets/demo/face_mesh.jpg" width="100%">

---

**UniFace** is a lightweight, production-ready Python library for face detection, recognition, tracking, landmark analysis, face parsing, gaze estimation, and face attributes.

GitHub: [github.com/yakhyo/uniface](https://github.com/yakhyo/uniface) | Docs: [yakhyo.github.io/uniface](https://yakhyo.github.io/uniface)

## Setup


In [ ]:
%pip install -q "uniface[cpu]"

# Clone repo for assets (Colab / Kaggle only)
import os
if any(k in os.environ for k in ('COLAB_GPU', 'COLAB_RELEASE_TAG', 'KAGGLE_KERNEL_RUN_TYPE')):
    if not os.path.exists('uniface'):
        !git clone --depth 1 https://github.com/yakhyo/uniface.git
    os.chdir('uniface/examples')

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

import uniface
from uniface.detection import SCRFD, BlazeFace
from uniface.draw import draw_mesh
from uniface.landmark import FaceMesh

print(f"UniFace version: {uniface.__version__}")

In [ ]:
# Any detector works — Face Mesh only needs a box plus the two eye points
detector = SCRFD()
mesher = FaceMesh()

print(f"Landmarks: {mesher.num_landmarks}")
print(f"Input size: {mesher.input_size}x{mesher.input_size}")

## 1. Detect and Mesh

Face Mesh needs a bounding box plus the first two landmarks (the eyes), which it uses to rotate the crop so the eye line is horizontal. Passing `Face` objects does this automatically.

In [ ]:
image = cv2.imread('../assets/source/mesh_face.jpg')

faces = detector.detect(image)
print(f"Detected {len(faces)} face(s)")

# One batched call for every face in the image
results = mesher.predict(image, faces)

result = results[0]
print(f"Landmarks shape: {result.landmarks.shape}")   # (468, 3)
print(f"2D only: {result.points_2d.shape}")   # (468, 2)
print(f"Presence score: {result.score:.4f}")

## 2. Visualize

`draw_mesh` has three render modes. `full` draws the dense 2556-edge tessellation, the most detailed but noticeably slower. Prefer `partial` or `points` for video.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, mode in zip(axes, ['points', 'partial', 'full']):
    canvas = image.copy()
    draw_mesh(canvas, result.landmarks, mode=mode)
    ax.imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
    ax.set_title(f"mode='{mode}'")
    ax.axis('off')

plt.tight_layout()
plt.show()

## 3. The Depth Coordinate

`z` is **relative** depth on the same pixel scale as `x`/`y`, where smaller is closer to the camera. It has no absolute origin, so it is only meaningful *within* one face, not between faces or between images.

In [ ]:
points = result.landmarks

plt.figure(figsize=(7, 7))
scatter = plt.scatter(points[:, 0], points[:, 1], c=points[:, 2], cmap='viridis', s=8)
plt.colorbar(scatter, label='z (relative depth, smaller = closer)')
plt.gca().invert_yaxis()
plt.gca().set_aspect('equal')
plt.title('468 landmarks coloured by depth')
plt.show()

print(f"z range: {points[:, 2].min():.1f} .. {points[:, 2].max():.1f}")
print("The nose tip is the closest point; the ears and jaw edges sit furthest back.")

## 4. The 478-Point Variant: Irises

`V2_478` returns the same 468 mesh points in the same order, plus ten iris points. Switching
is a one-word change, and region membership lives in named constants rather than magic indices.

In [ ]:
from uniface.constants import FaceMeshWeights
from uniface.landmark import IRIS_LEFT, IRIS_RIGHT, NUM_MESH_LANDMARKS

iris_mesher = FaceMesh(model_name=FaceMeshWeights.V2_478)
iris_result = iris_mesher.predict(image, faces)[0]

print(f"landmarks: {iris_result.landmarks.shape}")
print(f"the first {NUM_MESH_LANDMARKS} are exactly what V1_468 returns")
print(f"left iris: {iris_result.landmarks[IRIS_LEFT].shape} (centre, right, top, left, bottom)")
print(f"right iris: {iris_result.landmarks[IRIS_RIGHT].shape}")

left_pupil = iris_result.landmarks[IRIS_LEFT][0, :2]
print(f"left pupil at: ({left_pupil[0]:.0f}, {left_pupil[1]:.0f})")

# Each iris is centre-first, so the mean distance from point 0 to the other four is its radius.
# A human iris is close to 11.7 mm across whoever you measure, which turns that radius into
# a real-world scale reference.
iris = iris_result.landmarks[IRIS_LEFT]
radius_px = np.mean(np.linalg.norm(iris[1:, :2] - iris[0, :2], axis=1))
print(f"iris radius: {radius_px:.1f} px -> {11.7 / (2 * radius_px):.4f} mm per pixel")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

for ax, (res, title) in zip(axes, [(result, f"V1_468 - {len(result.landmarks)} points"),
                                   (iris_result, f"V2_478 - {len(iris_result.landmarks)} points")]):
    canvas = image.copy()
    draw_mesh(canvas, res.landmarks, mode="partial")
    # mark the ten iris points in red, so the difference is visible
    if len(res.landmarks) > NUM_MESH_LANDMARKS:
        # IRIS_LEFT / IRIS_RIGHT are slices, so index with them and iterate the points
        iris_points = np.vstack([res.landmarks[IRIS_LEFT], res.landmarks[IRIS_RIGHT]])
        for x, y in iris_points[:, :2]:
            cv2.circle(canvas, (int(x), int(y)), 4, (0, 0, 255), -1)
    ax.imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
    ax.set_title(title, fontsize=13)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 5. MediaPipe Parity

Seeding the mesh with **BlazeFace** reproduces MediaPipe's own pipeline exactly.

> ⚠️ BlazeFace returns **6** MediaPipe keypoints whose 4th point is a mouth *center*, not corners, so they cannot be fitted to the 5-point alignment template. It declares this with `supports_alignment = False`, and `FaceAnalyzer` disables recognition for it rather than producing broken embeddings.

In [ ]:
# MediaPipe runs BlazeFace ahead of Face Mesh in its own pipeline
blazeface = BlazeFace()

bf_faces = blazeface.detect(image)
bf_results = mesher.predict(image, bf_faces)

print(f"BlazeFace keypoints: {bf_faces[0].landmarks.shape}")       # (6, 2), not (5, 2)
print(f"supports_alignment: {blazeface.supports_alignment}")      # False

canvas = image.copy()
draw_mesh(canvas, bf_results[0].landmarks, mode='full')

plt.figure(figsize=(7, 7))
plt.imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
plt.title('MediaPipe parity: BlazeFace + Face Mesh')
plt.axis('off')
plt.show()

## 6. Other Ways to Call It

In [ ]:
# Face Mesh implements the same interface as Landmark106 and PIPNet
landmarks_2d = mesher.get_landmarks(image, faces[0].bbox)
print(f"get_landmarks: {landmarks_2d.shape}")

# Without a detector, pass boxes directly
x1, y1, x2, y2 = faces[0].bbox
manual = mesher.predict(image, bboxes=[[x1, y1, x2, y2]])
print(f"From a raw box: {manual[0].landmarks.shape}")

# Tune the crop with margin= if the mesh clips on unusual framing
tight = mesher.predict(image, faces, margin=0.1)[0]
print(f"Tighter crop shifts the fit: {not np.allclose(tight.landmarks, result.landmarks)}")

## Notes

- `z` is a relative depth on the same scale as `x` and `y`, not a distance in millimetres.
- The presence score saturates near 1.0, so it tells you the model ran rather than how well the
  mesh fits.
- `IRIS_LEFT` and `IRIS_RIGHT` are slices, not index lists. Index the array with them.
- Face Mesh and BlazeFace weights come from Google MediaPipe and are Apache-2.0 licensed.